# Explicit PRNG keys: split, never reuse

JAX has no global random state. Every random call takes a **key**, and the same key always gives the same numbers. To get fresh randomness you split a key into new subkeys, and each subkey is used exactly once.

Write `noisy_dropout_layer(seed, x, rate, sigma)` for `x` of shape `(B, D)`:

1. Make the key with `jax.random.PRNGKey(seed)` and split it **once** into three subkeys: `k_w, k_drop, k_noise = jax.random.split(key, 3)`.
2. Weights: `w = jax.random.normal(k_w, (D, D)) / sqrt(D)`, then `h = x @ w`.
3. Inverted dropout on `h` with `keep = jax.random.bernoulli(k_drop, 1 - rate, h.shape)`: kept entries are divided by `1 - rate`, dropped entries become `0`. `0 <= rate < 1`.
4. Per-row noise: split `k_noise` into `B` row keys with `jax.random.split(k_noise, B)`. Row `b` gets `jax.random.normal(row_keys[b], (D,))`. Add `sigma` times this noise to the dropout output.

Return `(w, out)`: float32 arrays of shapes `(D, D)` and `(B, D)`.

This problem runs in **Colab** (JAX isn't available in the browser). Use the default CPU runtime and a recent JAX (0.5 or later; the expected numbers come from JAX's default threefry generator). JAX computes in float32 here. The notebook includes the same tests.

Write your solution in the next cell, then run the test cell.

In [ ]:
import jax
import jax.numpy as jnp

def noisy_dropout_layer(seed, x, rate, sigma):
    """Return (w (D, D), out (B, D)) using keys split from jax.random.PRNGKey(seed)."""
    raise NotImplementedError


In [ ]:
# Tests (same harness and cases as the website)
_HARNESS = '"""Test harness shared by build.py (CPython) and the browser runner (Pyodide).\n\nOne file, so a test that passes at build time is graded the same way in the browser.\nValues cross JSON with small tags: ndarray, tuple, DataFrame/Series, non-finite floats.\n"""\nimport json, math\n\ntry:\n    import numpy as np\nexcept ImportError:  # pure-python problems never need it\n    np = None\n\n\ndef enc(x):\n    if np is not None and isinstance(x, np.ndarray):\n        return {\'__nd__\': enc(x.tolist()), \'dtype\': str(x.dtype)}\n    if np is not None and isinstance(x, np.generic):\n        return enc(x.item())\n    if isinstance(x, float) and not math.isfinite(x):\n        return {\'__f__\': repr(x)}\n    if isinstance(x, tuple):\n        return {\'__tup__\': [enc(v) for v in x]}\n    if isinstance(x, list):\n        return [enc(v) for v in x]\n    if isinstance(x, dict):\n        return {\'__dict__\': [[enc(k), enc(v)] for k, v in x.items()]}\n    t = type(x).__name__\n    if t == \'Tensor\':  # torch, in Colab and at build time\n        return enc(x.detach().cpu().numpy())\n    if t == \'DataFrame\':\n        # Cells go through enc() as Python values, so floats keep full precision (to_json caps at 15 digits).\n        return {\'__df__\': {\'columns\': [enc(c) for c in x.columns], \'data\': [enc(list(r)) for r in x.itertuples(index=False, name=None)]},\n                \'dtypes\': [str(d) for d in x.dtypes]}\n    if t == \'Series\':\n        return {\'__sr__\': enc(x.tolist()), \'name\': x.name}\n    return x\n\n\ndef dec(x):\n    if isinstance(x, list):\n        return [dec(v) for v in x]\n    if not isinstance(x, dict):\n        return x\n    if \'__nd__\' in x:\n        return np.array(dec(x[\'__nd__\']), dtype=x[\'dtype\'])\n    if \'__f__\' in x:\n        return float(x[\'__f__\'])\n    if \'__tup__\' in x:\n        return tuple(dec(v) for v in x[\'__tup__\'])\n    if \'__dict__\' in x:\n        return {_hashable(dec(k)): dec(v) for k, v in x[\'__dict__\']}\n    if \'__df__\' in x:\n        import pandas as pd\n        s = x[\'__df__\']\n        df = pd.DataFrame([dec(r) for r in s[\'data\']], columns=[dec(c) for c in s[\'columns\']])\n        for c, d in zip(df.columns, x[\'dtypes\']):\n            if d.startswith((\'int\', \'float\', \'bool\')):\n                df[c] = df[c].astype(d)\n        return df\n    if \'__sr__\' in x:\n        import pandas as pd\n        return pd.Series(dec(x[\'__sr__\']), name=x[\'name\'])\n    return x\n\n\ndef _hashable(k):\n    return tuple(k) if isinstance(k, list) else k\n\n\ndef same(got, exp, rtol=1e-6, atol=1e-8, path=\'output\'):\n    """(ok, why). Floats compare with tolerance, everything else exactly. Shapes must match."""\n    if type(exp).__name__ == \'DataFrame\':\n        if type(got).__name__ != \'DataFrame\':\n            return False, f\'{path}: expected a DataFrame, got {type(got).__name__}\'\n        if list(got.columns) != list(exp.columns):\n            return False, f\'{path}: columns {list(got.columns)} != expected {list(exp.columns)}\'\n        if got.shape != exp.shape:\n            return False, f\'{path}: shape {got.shape} != expected {exp.shape}\'\n        for c in exp.columns:\n            ok, why = same(got[c].tolist(), exp[c].tolist(), rtol, atol, f\'{path}[{c!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if type(exp).__name__ == \'Series\':\n        exp = exp.tolist()\n        got = got.tolist() if type(got).__name__ == \'Series\' else got\n    if np is not None and isinstance(exp, np.ndarray):\n        try:\n            g = np.asarray(got)\n        except Exception:\n            return False, f\'{path}: expected an array, got {type(got).__name__}\'\n        if g.shape != exp.shape:\n            return False, f\'{path}: shape {g.shape} != expected {exp.shape}\'\n        if exp.dtype.kind in \'fc\' or g.dtype.kind in \'fc\':\n            if g.dtype.kind not in \'fciub\':\n                return False, f\'{path}: expected numbers, got dtype {g.dtype}\'\n            bad = ~np.isclose(g.astype(complex if exp.dtype.kind == \'c\' else float), exp, rtol=rtol, atol=atol, equal_nan=True)\n            if bad.any():\n                i = tuple(int(v) for v in np.argwhere(bad)[0])\n                return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n            return True, \'\'\n        if not np.array_equal(g, exp):\n            i = tuple(int(v) for v in np.argwhere(g != exp)[0]) if g.shape else ()\n            return False, f\'{path}{list(i)}: {g[i].tolist()!r} != expected {exp[i].tolist()!r}\'\n        return True, \'\'\n    if isinstance(exp, float) or (isinstance(exp, int) and isinstance(got, float) and not isinstance(exp, bool)):\n        if np is not None and isinstance(got, np.generic):\n            got = got.item()\n        if not isinstance(got, (int, float)) or isinstance(got, bool):\n            return False, f\'{path}: expected a number, got {type(got).__name__}\'\n        if math.isnan(exp) and math.isnan(got):\n            return True, \'\'\n        return (True, \'\') if math.isclose(got, exp, rel_tol=rtol, abs_tol=atol) else (False, f\'{path}: {got!r} != expected {exp!r}\')\n    if isinstance(exp, (list, tuple)):\n        if np is not None and isinstance(got, np.ndarray):\n            got = got.tolist()\n        if not isinstance(got, (list, tuple)):\n            return False, f\'{path}: expected a {type(exp).__name__}, got {type(got).__name__}\'\n        if len(got) != len(exp):\n            return False, f\'{path}: length {len(got)} != expected {len(exp)}\'\n        for i, (g, e) in enumerate(zip(got, exp)):\n            ok, why = same(g, e, rtol, atol, f\'{path}[{i}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if isinstance(exp, dict):\n        if not isinstance(got, dict):\n            return False, f\'{path}: expected a dict, got {type(got).__name__}\'\n        if set(got) != set(exp):\n            return False, f\'{path}: keys {sorted(map(str, got))} != expected {sorted(map(str, exp))}\'\n        for k in exp:\n            ok, why = same(got[k], exp[k], rtol, atol, f\'{path}[{k!r}]\')\n            if not ok:\n                return ok, why\n        return True, \'\'\n    if np is not None and isinstance(got, np.generic):\n        got = got.item()\n    return (True, \'\') if got == exp else (False, f\'{path}: {got!r} != expected {exp!r}\')\n\n\ndef run_sql(query, setup):\n    """Run `setup` (CREATE/INSERT script) in a fresh in-memory db, then `query`; rows as lists."""\n    import sqlite3  # a separate package in Pyodide: only SQL problems load it\n    con = sqlite3.connect(\':memory:\')\n    try:\n        con.executescript(setup)\n        cur = con.execute(query)\n        return [[c[0] for c in cur.description], [list(r) for r in cur.fetchall()]]\n    finally:\n        con.close()\n\n\ndef show_args(args, runtime, n=600):\n    """Arguments as the learner would write them: one per line, SQL setup as-is."""\n    if runtime == \'sql\':\n        return args[0] if len(args[0]) <= n else args[0][:n] + \' …\'\n    return \'\\n\'.join(short(a, n) for a in args)\n\n\ndef short(x, n=300):\n    s = repr(x)\n    return s if len(s) <= n else s[:n] + \' …\'\n\n\ndef grade(fn, tests, runtime, rtol=1e-6, atol=1e-8):\n    """Run `fn` (a callable, or SQL text for runtime \'sql\') on each test; list of result dicts."""\n    out = []\n    for t in tests:\n        args, exp = prep(dec(t[\'a\']), runtime), dec(t[\'e\'])\n        try:\n            got = run_sql(fn, args[0]) if runtime == \'sql\' else plain(fn(*_copy(args)))\n            ok, why = same(got, exp, rtol, atol)\n        except Exception as e:  # learner code raised: report it like a failed test\n            got, ok, why = None, False, f\'{type(e).__name__}: {e}\'\n        out.append({\'ok\': ok, \'why\': why, \'got\': short(got) if not ok else \'\',\n                    \'inp\': show_args(args, runtime, 400) if not ok else \'\'})\n    return out\n\n\ndef prep(args, runtime):\n    """Arrays cross JSON as ndarrays; torch problems take tensors."""\n    if runtime == \'torch\':\n        import torch\n        return [torch.as_tensor(a) if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    if runtime == \'triton\':                                  # Colab only: kernels need CUDA tensors\n        import torch\n        return [torch.as_tensor(a).cuda() if np is not None and isinstance(a, np.ndarray) else a for a in args]\n    return args\n\n\ndef plain(x):\n    """Framework arrays (torch, jax) -> numpy, recursively, so results compare and encode the same way."""\n    t = type(x).__name__\n    if t == \'Tensor\':\n        return x.detach().cpu().numpy()\n    if t in (\'ArrayImpl\', \'Array\') and hasattr(x, \'__array__\'):\n        return np.asarray(x)\n    if isinstance(x, tuple):\n        return tuple(plain(v) for v in x)\n    if isinstance(x, list):\n        return [plain(v) for v in x]\n    if isinstance(x, dict):\n        return {k: plain(v) for k, v in x.items()}\n    return x\n\n\ndef _copy(args):\n    # Tests reuse inputs across Run and Submit, so a function that mutates its input mustn\'t poison later cases.\n    import copy\n    return copy.deepcopy(args)\n'
_TESTS = '[{"a": [0, {"__nd__": [[1.0, 1.0, 1.0], [1.0, 1.0, 1.0]], "dtype": "float64"}, 0.0, 0.0], "e": {"__tup__": [{"__nd__": [[0.579667866230011, -0.5232740044593811, -0.4319574236869812], [-0.6762890219688416, -0.5030064582824707, 0.3399658203125], [0.41796115040779114, -0.5921293497085571, 0.9593413472175598]], "dtype": "float32"}, {"__nd__": [[0.32133999466896057, -1.6184097528457642, 0.8673497438430786], [0.32133999466896057, -1.6184097528457642, 0.8673497438430786]], "dtype": "float32"}]}}, {"a": [1, {"__nd__": [[1.0, -1.0], [0.5, 2.0]], "dtype": "float64"}, 0.5, 0.1], "e": {"__tup__": [{"__nd__": [[0.37613651156425476, -0.5112502574920654], [0.6415275931358337, -0.6720714569091797]], "dtype": "float32"}, {"__nd__": [[-0.36842089891433716, -0.2385975569486618], [3.098616600036621, -3.2803404331207275]], "dtype": "float32"}]}}, {"a": [7, {"__nd__": [[-0.8019314252534474, -1.324358995628145, -0.24836162209524854, 0.4204452380655215, 1.1360465324896427], [0.10970639932180819, -0.5526473205362324, -0.7847803553442784, 0.7487457707345911, 1.6347830429585775], [0.27276877584472176, -1.2333286640307717, -0.9582652054360887, 1.6000190889991115, 0.2028824405086084], [-1.7321348424395848, -0.08369619281702581, -1.1632259734447485, -0.6292880940615545, -0.48800582327685743]], "dtype": "float64"}, 0.25, 0.0], "e": {"__tup__": [{"__nd__": [[0.49821704626083374, 0.06333943456411362, -0.2336486130952835, -0.19366416335105896, 1.078902244567871], [-0.36737415194511414, -0.07893966138362885, -0.40901312232017517, 0.4554688632488251, -0.8426018953323364], [-0.5303654074668884, 0.4424401521682739, 0.23583300411701202, -0.4737551808357239, 0.22417768836021423], [0.18842852115631104, 0.19966518878936768, -0.4039798676967621, -0.022337250411510468, 0.19434136152267456], [0.04479833319783211, 0.49266964197158813, -0.1789265275001526, 0.5098851919174194, 0.22214943170547485]], "dtype": "float32"}, {"__nd__": [[0.4651181399822235, 0.0, 0.3964775800704956, 0.3195006251335144, 0.7054754495620728], [1.1843026876449585, 0.0, 0.0, 0.0, 1.222362995147705], [1.8770664930343628, 0.14677694439888, -0.6239354014396667, 0.0, 1.9662590026855469], [0.0, -1.311781406402588, 0.6748714447021484, 0.8181939125061035, -3.0530104637145996]], "dtype": "float32"}]}}, {"a": [7, {"__nd__": [[-0.7133133716322436, 0.5533784703532895, -0.06308597192528916, -0.5894312580326048, 0.40963782655711695], [0.8298553070613239, -1.643023371405677, -0.256730126365494, -0.9807473560440125, -0.17315522486203205], [-1.2894187467538587, 0.0206903940375912, -0.03788574104406823, -0.304337750958489, -1.0479265051202462], [-0.3961903304730927, -1.091328901695709, -1.3552087462047395, 0.22478573245989314, -1.109349937891366]], "dtype": "float64"}, 0.25, 1.0], "e": {"__tup__": [{"__nd__": [[0.49821704626083374, 0.06333943456411362, -0.2336486130952835, -0.19366416335105896, 1.078902244567871], [-0.36737415194511414, -0.07893966138362885, -0.40901312232017517, 0.4554688632488251, -0.8426018953323364], [-0.5303654074668884, 0.4424401521682739, 0.23583300411701202, -0.4737551808357239, 0.22417768836021423], [0.18842852115631104, 0.19966518878936768, -0.4039798676967621, -0.022337250411510468, 0.19434136152267456], [0.04479833319783211, 0.49266964197158813, -0.1789265275001526, 0.5098851919174194, 0.22214943170547485]], "dtype": "float32"}, {"__nd__": [[0.085529625415802, -0.5181703567504883, -0.9221413135528564, 0.7387692332267761, -1.7648930549621582], [2.7379202842712402, -0.6243452429771423, -0.6382796764373779, -1.1197260618209839, 0.9406706094741821], [-0.5488399863243103, -2.5859646797180176, 0.6651054620742798, 1.563849687576294, -2.8475184440612793], [-0.44476890563964844, -1.072595477104187, 0.4580341875553131, -0.07377547025680542, -0.06731899082660675]], "dtype": "float32"}]}}, {"a": [123, {"__nd__": [[0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0], [0.0, 0.0, 0.0, 0.0]], "dtype": "float64"}, 0.9, 0.5], "e": {"__tup__": [{"__nd__": [[-0.24546678364276886, -0.4739346504211426, -0.8875985145568848, -0.15050160884857178], [0.02468053624033928, -0.25578171014785767, -0.3307657539844513, -0.07565949112176895], [0.34267544746398926, 1.1463849544525146, 0.33544740080833435, -0.11893097311258316], [-0.16479629278182983, -0.3816324472427368, 0.680259644985199, 0.41964268684387207]], "dtype": "float32"}, {"__nd__": [[-0.28108012676239014, -0.7900061011314392, -0.44935229420661926, -0.5671717524528503], [-0.3244289755821228, 0.8134971857070923, 0.32148629426956177, 0.0003771398332901299], [1.0318760871887207, -0.8140677213668823, 0.3778845965862274, 0.22157315909862518]], "dtype": "float32"}]}}, {"a": [2024, {"__nd__": [[1.1702961011782933, 0.7165876558738361, -1.9978166924497212, 0.272128869412488, -1.1017166275810448, 0.033057220158269195, 0.04363199256942161, -1.9884297882311208], [-0.23342252376577002, -0.255790031399391, 0.9620005318430944, -1.1814468079562157, 0.7380418978456841, -1.0989727630364063, -0.33129089269991674, -0.8404731684222111], [1.448731288921672, 0.5682130997882933, 2.4317325028452124, 0.6419163790823205, 0.8449927337191754, 0.8406828762653401, -0.6066115359095516, -0.07002844663838208], [1.350388867744626, -0.3965507651729716, 0.18879953129109867, -0.021223460417289796, 0.6092164928327407, -0.3649087419473726, -0.15236188875684828, 0.24238142867215043], [0.10302313848680769, -0.8649727462818201, 0.8957830431894438, -1.298481208246912, -1.2011155488035266, -1.282491794740462, 0.9669722789332148, -0.36060836889927], [-0.9710363785210655, -1.1360213941896466, 0.42113113746240616, -1.054840662577835, -1.2720782100976422, 0.6139930624688609, -1.1967077271925706, -0.32243815075139726]], "dtype": "float64"}, 0.1, 0.05], "e": {"__tup__": [{"__nd__": [[0.14610697329044342, 0.38157761096954346, 0.49419325590133667, 0.42137134075164795, -0.7161422967910767, 0.35529038310050964, -0.34671550989151, 0.3315359652042389], [-0.19720906019210815, -0.07814507186412811, 0.019208500161767006, -0.7904666662216187, -0.13548414409160614, 0.36185723543167114, 0.0922526940703392, 0.3962514400482178], [-0.6299766898155212, -0.005583939142525196, 0.15092803537845612, 0.5861289501190186, 0.30902299284935, 0.4240652322769165, -0.06267447769641876, 0.1590663343667984], [0.04333789274096489, 0.38794660568237305, -0.05585751682519913, -0.11091899871826172, 0.38579651713371277, 0.6378419399261475, 0.0007981862290762365, -0.1967964470386505], [0.7451132535934448, 0.27291232347488403, -0.36075496673583984, 0.3438440263271332, -0.13837279379367828, -0.4070948362350464, 0.5859015583992004, -0.13501596450805664], [-0.7697165012359619, -0.07374552637338638, 0.3896850049495697, -0.08747729659080505, 0.2514691948890686, 0.17514684796333313, -0.2877771258354187, -0.5647823214530945], [-0.23179921507835388, 0.11588743329048157, 0.24974960088729858, 0.4626910090446472, 0.3042762875556946, 0.16382625699043274, 0.018155213445425034, 0.15973281860351562], [-0.392499178647995, -0.8670772314071655, -0.366371750831604, 0.02798369526863098, 0.06181949004530907, 0.3314477503299713, 0.2540176212787628, 0.1405162364244461]], "dtype": "float32"}, {"__nd__": [[1.361173391342163, 2.177647113800049, 1.5912928581237793, -1.7640892267227173, -1.6066875457763672, -0.013931715860962868, -1.543578028678894, 0.15083715319633484], [1.2061998844146729, 0.4888005554676056, -0.34635236859321594, 1.1361687183380127, -0.4954681992530823, -1.3732938766479492, 0.5275863409042358, -0.040712982416152954], [-0.001924530602991581, 0.9320387244224548, 1.0307974815368652, 1.658624529838562, -0.19266629219055176, 2.0440404415130615, -0.33948230743408203, 0.2502497434616089], [0.8616480827331543, 0.5275005102157593, 0.1808232069015503, 1.286339282989502, -1.1529555320739746, 0.1595430076122284, 0.028369886800646782, 0.47457197308540344], [0.018649961799383163, -0.2132200002670288, 0.5876463055610657, 1.8151812553405762, 0.03224261850118637, -0.418583482503891, -0.6266775131225586, 1.231324553489685], [-1.3388158082962036, -1.0820879936218262, 0.165068119764328, 0.09853999316692352, 0.6438076496124268, -0.9920209646224976, -0.036638110876083374, -0.9333822131156921]], "dtype": "float32"}]}}, {"a": [42, {"__nd__": [[-0.006761540346521785, -0.4453353669298123, -0.05409396159544828, 1.3387734990547715, -0.516894152589275, -1.2593071588782276, -1.8367457051936795, -0.20476611683386706, -0.3522570104112305, 0.2650909069303235, -0.4642445916374816, -0.4786384630527245, -0.7213157478486046, -0.5197597723701788, 0.1602267063174243, -0.3803527272151388]], "dtype": "float64"}, 0.0, 2.0], "e": {"__tup__": [{"__nd__": [[0.018981384113430977, -0.1215856596827507, 0.32258015871047974, 0.12990297377109528, 0.0751010924577713, 0.07758716493844986, 0.14404022693634033, -0.2018655240535736, -0.49708041548728943, 0.15988238155841827, 0.054407935589551926, 0.0006185629172250628, 0.41614264249801636, 0.05078383907675743, -0.005345562007278204, -0.17169761657714844], [0.004806129727512598, 0.11789745837450027, -0.21020133793354034, 0.2551800310611725, -0.5250747799873352, 0.1077471598982811, -0.17268744111061096, 0.27073943614959717, 0.049112409353256226, 0.2126951664686203, 0.45445364713668823, 0.17842449247837067, 0.327706903219223, -0.39531853795051575, 0.0003644216922111809, -0.06487097591161728], [0.24009844660758972, 0.10607295483350754, -0.16291673481464386, 0.23160384595394135, -0.08052758872509003, -0.11294283717870712, -0.3381143808364868, 0.11769387125968933, 0.07500895857810974, -0.28125736117362976, -0.023461490869522095, 0.5391325950622559, 0.07500231266021729, -0.0007450735429301858, -0.06879261136054993, -0.1232258528470993], [0.035344820469617844, 0.07789191603660583, 0.004810388665646315, 0.4938233196735382, -0.06340396404266357, 0.4537223279476166, 0.21700380742549896, 0.17500244081020355, -0.03214171901345253, 0.0007112701423466206, -0.0006201690412126482, -0.13286332786083221, 0.4121800363063812, 0.5071414113044739, -0.0008843212272040546, -0.18373532593250275], [0.33663153648376465, -0.04741910099983215, -0.5716466307640076, 0.21762259304523468, 0.0858025923371315, 0.5925543308258057, 0.23930762708187103, 0.11606477200984955, -0.022409431636333466, -0.3614269495010376, 0.18344151973724365, 0.01698320545256138, 0.004494712222367525, -0.2011176347732544, -0.06472565978765488, -0.060990702360868454], [-0.10760335624217987, 0.6645383238792419, -0.20468422770500183, 0.11878110468387604, -0.2712985575199127, -0.19294138252735138, -0.002887105569243431, 0.4406975209712982, 0.10186149924993515, -0.06750838458538055, 0.21858343482017517, -0.11082526296377182, 0.1930680125951767, -0.0961630716919899, 0.05144359543919563, -0.0031780966091901064], [-0.21924760937690735, 0.19467316567897797, -0.1520347148180008, 0.10085007548332214, -0.11763707548379898, 0.13804979622364044, -0.01510776299983263, 0.05551280453801155, 0.1899499148130417, -0.08306080847978592, 0.0014889398589730263, -0.31989073753356934, 0.10084010660648346, 0.37909260392189026, 0.517694354057312, -0.03844835236668587], [-0.13408263027668, -0.217483788728714, 0.4070781469345093, -0.008989925496280193, -0.12492316216230392, 0.10174471139907837, -0.0011856023920699954, -0.3123592734336853, 0.442411333322525, 0.010295404121279716, -0.5555246472358704, -0.1028788611292839, -0.31068912148475647, -0.5703455805778503, -0.468292772769928, -0.19687840342521667], [0.15409831702709198, -0.3501541018486023, 0.11645487695932388, -0.3774532079696655, 0.3488835394382477, -0.5194730758666992, 0.21887043118476868, -0.10901525616645813, 0.005658383015543222, 0.1948433667421341, 0.19651135802268982, 0.20369040966033936, -0.085633784532547, -0.04160512238740921, -0.2643434703350067, 0.16135206818580627], [0.1287839561700821, -0.15070801973342896, 0.38175562024116516, -0.27144595980644226, 0.15779957175254822, -0.10350734740495682, 0.00011132664803881198, -0.017759714275598526, -0.24930831789970398, 0.1697489321231842, -0.17604480683803558, 0.5043909549713135, -0.3243202269077301, -0.30008935928344727, -0.08723637461662292, 0.15091775357723236], [0.11327165365219116, -0.22762802243232727, 0.24882663786411285, -0.163376122713089, 0.10997220873832703, 0.028363756835460663, 0.12470134347677231, 0.5146337747573853, 0.07213687151670456, 0.14930416643619537, 0.10646037012338638, 0.06755506247282028, -0.016638007014989853, -0.11048226803541183, 0.1146153062582016, 0.3227449059486389], [-0.08302132785320282, 0.5066983699798584, 0.28122493624687195, -0.07364791631698608, -0.12647971510887146, -0.004826489835977554, -0.4049273133277893, -0.11878608912229538, -0.2056380808353424, -0.13829421997070312, -0.2594572901725769, -0.3227733373641968, -0.4114193320274353, 0.12086520344018936, 0.14381371438503265, 0.12285926938056946], [0.3967640697956085, 0.1174771711230278, 0.030982572585344315, -0.14718860387802124, -0.23322083055973053, -0.05604144185781479, -0.30799195170402527, -0.03832188993692398, -0.33293819427490234, -0.47033727169036865, 0.1136711984872818, 0.27871793508529663, 0.3611909747123718, -0.21648064255714417, -0.3971608281135559, -0.2690432369709015], [-0.11309871822595596, 0.06407099217176437, -0.15743803977966309, 0.2936447262763977, -0.2744086980819702, 0.11538857221603394, -0.3014400005340576, -0.47365686297416687, 0.03233391419053078, 0.6277086138725281, 0.08762618154287338, 0.39774540066719055, -0.35887303948402405, 0.006777039263397455, 0.45890146493911743, 0.02519063837826252], [0.22315776348114014, -0.07136251777410507, 0.23157678544521332, -0.14330749213695526, 0.3203359544277191, -0.345729798078537, -0.06088681146502495, 0.3886420726776123, -0.26106318831443787, 0.04234332591295242, -0.39774924516677856, -0.011354458518326283, -0.028118401765823364, -0.39122819900512695, -0.15389499068260193, -0.09725072234869003], [-0.4003603160381317, 0.04418555647134781, 0.059777338057756424, -0.0982675775885582, -0.1615222692489624, -0.5384085178375244, -0.21954086422920227, -0.07469320297241211, 0.5804859399795532, -0.1534998118877411, 0.0007259867852553725, -0.11724402010440826, 0.6961323618888855, 0.031044451519846916, 0.060170985758304596, 0.011383884586393833]], "dtype": "float32"}, {"__nd__": [[-0.6105265021324158, 1.2985042333602905, 0.533345103263855, -1.2896736860275269, 2.3154373168945312, 2.453758955001831, 1.1281169652938843, -0.9815578460693359, 0.2514130473136902, -1.9822807312011719, -0.05264860391616821, -3.1549594402313232, 0.42360639572143555, 1.1743865013122559, -1.3781678676605225, 0.9878380298614502]], "dtype": "float32"}]}}]'
import json, types
harness = types.ModuleType('harness'); exec(_HARNESS, harness.__dict__)
res = harness.grade(noisy_dropout_layer, json.loads(_TESTS), 'jax', *[0.0001, 1e-05])
for i, r in enumerate(res): print(f"test {i+1}: {'pass' if r['ok'] else 'FAIL  ' + r['why']}")
print(f"{sum(r['ok'] for r in res)}/{len(res)} passed")